In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import datetime
import tqdm

print(datetime.datetime.now().isoformat())

2025-06-29T23:13:22.212559


In [3]:
import bioacoustics_model_zoo as bmz

2025-06-29 23:13:37.180473: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2025-06-29 23:13:37.198748: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:485] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2025-06-29 23:13:37.219611: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:8454] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2025-06-29 23:13:37.226235: E external/local_xla/xla/stream_executor/cuda/cuda_blas.cc:1452] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2025-06-29 23:13:37.243508: I tensorflow/core/platform/cpu_feature_guar

In [4]:
from pathlib import Path

audio = sorted(Path("~/scratch/birdclef/raw/birdclef-2024/train_audio/asbfly").expanduser().glob("*.ogg"))[
    :2
]
audio

[PosixPath('/storage/home/hcoda1/7/acheung46/scratch/birdclef/raw/birdclef-2024/train_audio/asbfly/XC134896.ogg'),
 PosixPath('/storage/home/hcoda1/7/acheung46/scratch/birdclef/raw/birdclef-2024/train_audio/asbfly/XC164848.ogg')]

In [5]:
import os
from transformers import AutoModel, AutoConfig
# from bioacoustics_model_zoo.BirdSetEfficientNetB1 import BirdSetEfficientNetB1 # Assuming you have this locally

model_dir = "/storage/coda1/p-dsgt_clef2025/0/shared/birdclef/models/2025/v1/BirdSetEfficientNetB1/torch-linear-v1"
os.makedirs(model_dir, exist_ok=True)

model_id = "DBD-research-group/EfficientNet-B1-BirdSet-XCL"

model = AutoModel.from_pretrained(model_id)
config = AutoConfig.from_pretrained(model_id)

model.save_pretrained(model_dir)
config.save_pretrained(model_dir)


config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/76.3M [00:00<?, ?B/s]

In [21]:
from opensoundscape import SpectrogramClassifier
from bioacoustics_model_zoo.bmz_birdset.bmz_birdset_efficientnetB1 import EfficientnetBirdsetPreprocessor, EfficientNetLogits

@bmz.register_bmz_model
class BirdSetEfficientNetB1Offline(SpectrogramClassifier):
    def __init__(self, model_path):
        """BirdSetEfficientNetB1 for offline Kaggle use"""

        model = EfficientNetLogits.from_pretrained(
            model_path,
            ignore_mismatched_sizes=True,
        )
        classes = [model.config.id2label[i] for i in range(model.num_labels)]

        super().__init__(model, classes=classes, sample_duration=5)

        self.preprocessor = EfficientnetBirdsetPreprocessor()
        self.network.to(self.device)

        self.network.classifier_layer = "classifier"
        self.network.embedding_layer = "efficientnet.pooler"
        self.network.cam_layer = "efficientnet.encoder.blocks.1"

In [22]:
birdset_efficientnet = BirdSetEfficientNetB1Offline(model_dir)

Some weights of EfficientNetLogits were not initialized from the model checkpoint at /storage/coda1/p-dsgt_clef2025/0/shared/birdclef/models/2025/v1/BirdSetEfficientNetB1/torch-linear-v1 and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
/storage/home/hcoda1/7/acheung46/scratch/birdclef/.venv/lib64/python3.9/site-packages/opensoundscape/ml/cnn.py:599: UserWarning: 
                    This architecture is not listed in opensoundscape.ml.cnn_architectures.ARCH_DICT.
                    It will not be available for loading after saving the model with .save() (unless using pickle=True). 
                    To make it re-loadable, define a function that generates the architecture from arguments: (n_classes, n_channels) 
                    then use opensoundscape.ml.cnn_architectures.register_architecture() to register the generating function.

                  

In [17]:
%time _ = birdset_efficientnet.predict(audio)

  0%|          | 0/8 [00:00<?, ?it/s]

CPU times: user 4.02 s, sys: 101 ms, total: 4.13 s
Wall time: 588 ms


In [18]:
%time _ = birdset_efficientnet.predict(audio, clip_step=1)

  0%|          | 0/34 [00:00<?, ?it/s]

CPU times: user 18.1 s, sys: 606 ms, total: 18.7 s
Wall time: 2.61 s


In [24]:
birdset_efficientnet.embed(audio)

  0%|          | 0/8 [00:00<?, ?it/s]

0     \
file                                               start_time end_time             
/storage/home/hcoda1/7/acheung46/scratch/birdcl... 0.0        5.0      -0.118963   
                                                   5.0        10.0     -0.049138   
                                                   10.0       15.0      0.302333   
                                                   15.0       20.0      0.038898   
                                                   20.0       25.0      0.060524   
/storage/home/hcoda1/7/acheung46/scratch/birdcl... 0.0        5.0      -0.085576   
                                                   5.0        10.0     -0.182065   
                                                   10.0       15.0     -0.164357   

                                                                            1     \
file                                               start_time end_time             
/storage/home/hcoda1/7/acheung46/scratch/birdcl... 0.0        5.0      -0.195107   
                                                   5.0        10.0     -0.168622   
                                                   10.0       15.0     -0.087450   
                                                   15.0       20.0     -0.137079   
                                                   20.0       25.0     -0.183898   
/storage/home/hcoda1/7/acheung46/scratch/birdcl... 0.0        5.0      -0.164860   
                                                   5.0        10.0     -0.178692   
                                                   10.0       15.0     -0.166790   

                                                                            2     \
file                                               start_time end_time             
/storage/home/hcoda1/7/acheung46/scratch/birdcl... 0.0        5.0      -0.158132   
                                                   5.0        10.0     -0.120275   
                                                   10.0       15.0     -0.105236   
                                                   15.0       20.0     -0.178694   
                                                   20.0       25.0     -0.127764   
/storage/home/hcoda1/7/acheung46/scratch/birdcl... 0.0        5.0       0.118055   
                                                   5.0        10.0     -0.100275   
                                                   10.0       15.0     -0.143708   

                                                                            3     \
file                                               start_time end_time             
/storage/home/hcoda1/7/acheung46/scratch/birdcl... 0.0        5.0      -0.038108   
                                                   5.0        10.0     -0.034843   
                                                   10.0       15.0      0.121466   
                                                   15.0       20.0      0.034435   
                                                   20.0       25.0     -0.187966   
/storage/home/hcoda1/7/acheung46/scratch/birdcl... 0.0        5.0      -0.149063   
                                                   5.0        10.0     -0.120030   
                                                   10.0       15.0     -0.112905   

                                                                            4     \
file                                               start_time end_time             
/storage/home/hcoda1/7/acheung46/scratch/birdcl... 0.0        5.0      -0.175408   
                                                   5.0        10.0     -0.100576   
                                                   10.0       15.0      0.024819   
                                                   15.0       20.0     -0.182235   
                                                   20.0       25.0     -0.133845   
/storage/home/hcoda1/7/acheung46/scratch/birdcl... 0.0        5.0      -0.184592   
                                        